<a href="https://colab.research.google.com/github/cdascientist/Vis/blob/master/vis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 1: ARCHITECTURE, MODELS, TELEMETRY
from __future__ import annotations
import datetime, pytz, os, logging, ctypes, gc, time, sys
import numpy as np
import plotly.graph_objects as go
from dataclasses import dataclass, field
from typing import Optional, Any, Callable


def denver_time_factory():
    denver_tz = pytz.timezone("America/Denver")
    def make_timestamp():
        now_utc    = datetime.datetime.now(pytz.utc)
        now_denver = now_utc.astimezone(denver_tz)
        gmt_stamp  = now_utc.strftime("%H:%M:%S GMT %B %d, %Y")
        local_stamp = (
            f"{now_denver.hour % 12 or 12}:"
            f"{now_denver.strftime('%M.%S.')}"
            f"{now_denver.strftime('%f')[:2]}."
            f"{now_denver.strftime('%f')[2:4]} "
            f"{now_denver.strftime('%p %B %d, %Y')}"
        )
        return gmt_stamp, local_stamp
    return make_timestamp


def setup_logging(log_filename: str = "ollama_installer.log") -> tuple[logging.Logger, str]:
    log_path = os.path.join("/content", log_filename)
    logger   = logging.getLogger("OllamaInstaller")
    logger.setLevel(logging.DEBUG)
    if not logger.handlers:
        fmt = logging.Formatter("%(asctime)s  %(name)s  %(levelname)s  %(message)s")
        fh  = logging.FileHandler(log_path);  fh.setLevel(logging.DEBUG);  fh.setFormatter(fmt)
        ch  = logging.StreamHandler();        ch.setLevel(logging.INFO);   ch.setFormatter(fmt)
        logger.addHandler(fh);  logger.addHandler(ch)
        logger.info(f"Log file created at: {log_path}")
    return logger, log_path


# ══════════════════════════════════════════════════════════════════════════════
#  UNIQUE RUNTIME ID GENERATOR
# ══════════════════════════════════════════════════════════════════════════════
class UIDGen:
    _counter: int = 0
    @staticmethod
    def next(prefix: str = "UID") -> str:
        UIDGen._counter += 1
        ts = datetime.datetime.now().strftime("%H%M%S%f")
        return f"{prefix}-{ts}-{UIDGen._counter:06d}"


# ══════════════════════════════════════════════════════════════════════════════
#  PACKAGE SCAN DATA  —  Denver Hub  2026-03-03
# ══════════════════════════════════════════════════════════════════════════════
_BASE = datetime.datetime(2026, 3, 3, 8, 0, 0)

def _ts(s: str) -> datetime.datetime:
    return datetime.datetime.strptime(s, "%Y-%m-%d %H:%M:%S")

def _offset(s: str) -> float:
    return (_ts(s) - _BASE).total_seconds() / 60.0

PKG001_SCANS = [
    {"scan_id": "5c8d30e1-4674-4cf0-9df6-ecbff2e45e93",
     "station": "Inbound Dock",        "timestamp": "2026-03-03 08:00:00",
     "location": "Denver Hub",         "offset_min": _offset("2026-03-03 08:00:00")},
    {"scan_id": "71599ed8-e781-4283-9d67-ee595b4a7d8b",
     "station": "Initial Scan",        "timestamp": "2026-03-03 08:11:00",
     "location": "Denver Hub",         "offset_min": _offset("2026-03-03 08:11:00")},
    {"scan_id": "4bcf7487-cfcb-4209-b93c-fe07a0731dcb",
     "station": "Sorting Belt A",      "timestamp": "2026-03-03 08:19:00",
     "location": "Denver Hub",         "offset_min": _offset("2026-03-03 08:19:00")},
    {"scan_id": "3f961294-316a-4367-9d35-7dcb77aee292",
     "station": "Convergence Station", "timestamp": "2026-03-03 08:36:00",
     "location": "Denver Hub",         "offset_min": _offset("2026-03-03 08:36:00")},
    {"scan_id": "1203df9d-e783-4105-a45a-2eb53274d772",
     "station": "Outbound Dock",       "timestamp": "2026-03-03 08:55:00",
     "location": "Denver Hub",         "offset_min": _offset("2026-03-03 08:55:00")},
]

PKG002_SCANS = [
    {"scan_id": "01d9cfd3-4c57-4e11-a0db-180ef86d753c",
     "station": "Inbound Dock",        "timestamp": "2026-03-03 08:03:00",
     "location": "Denver Hub",         "offset_min": _offset("2026-03-03 08:03:00")},
    {"scan_id": "d736de89-3aba-421a-a569-a15f41e53499",
     "station": "Initial Scan",        "timestamp": "2026-03-03 08:15:00",
     "location": "Denver Hub",         "offset_min": _offset("2026-03-03 08:15:00")},
    {"scan_id": "44803c72-a6c4-48eb-a7be-213d0d6e5763",
     "station": "Sorting Belt B",      "timestamp": "2026-03-03 08:32:00",
     "location": "Denver Hub",         "offset_min": _offset("2026-03-03 08:32:00")},
    {"scan_id": "9b7e822a-1c46-44fc-9566-5dca8c74a78a",
     "station": "Convergence Station", "timestamp": "2026-03-03 08:41:00",
     "location": "Denver Hub",         "offset_min": _offset("2026-03-03 08:41:00")},
    {"scan_id": "ec85e74e-7135-4d54-85b5-f9e793bf7055",
     "station": "Outbound Dock",       "timestamp": "2026-03-03 08:52:00",
     "location": "Denver Hub",         "offset_min": _offset("2026-03-03 08:52:00")},
]


# ══════════════════════════════════════════════════════════════════════════════
#  TANDEM LOCK CALCULATION LIBRARY
#
#  The TandemLock always has access to these functions.
#  Add new calculation functions here; reference them in HF rules via "fn": TandemCalc.xxx
#
#  TODO: add scan-time delta:
#    @staticmethod
#    def scan_delta(a, b): return round(abs(b - a), 4)
#
#  TODO: add throughput rate:
#    @staticmethod
#    def throughput(scan_count, elapsed_min): return round(scan_count / elapsed_min, 4) if elapsed_min else 0.0
#
#  TODO: add convergence efficiency (product / sum):
#    @staticmethod
#    def conv_efficiency(a, b): return round((a * b) / (a + b), 4) if (a + b) else 0.0
#
#  TODO: add normalised delta (gap as fraction of larger value):
#    @staticmethod
#    def norm_delta(a, b): return round(abs(b - a) / max(a, b, 1), 4)
# ══════════════════════════════════════════════════════════════════════════════
class TandemCalc:
    @staticmethod
    def multiply(a, b):   return round(a * b, 4)
    @staticmethod
    def gap(a, b):        return round(abs(b - a), 4)
    @staticmethod
    def ratio(a, b):      return round(a / b, 4) if b != 0 else 0.0
    @staticmethod
    def mean(a, b):       return round((a + b) / 2, 4)
    @staticmethod
    def weighted_sum(a, b, w_a=0.6, w_b=0.4): return round(a * w_a + b * w_b, 4)
    @staticmethod
    def lock_seal(a, b):  return f"LOCK::({a})||({b})"


# ══════════════════════════════════════════════════════════════════════════════
#  HF EXCHANGE CONFIGURATION
#
#  TWO SEPARATE PROPERTY SPACES:
#
#  1. HF NUMERIC CHAIN  — hf_value_a / hf_value_b / hf_result_p1 …
#     Simple numeric values.  TandemCalc functions operate on these.
#     Chain:  PRI(2) × SEC(3) = 6  →  ×4 = 24  →  ×2 = 48
#
#  2. PACKAGE SCAN FEATURES  — scan_id, station, timestamp, offset_min
#     Real facility scan data loaded onto each probe as additional features.
#     Carried independently; NOT merged by HF engine.  Used for later analysis.
#
#  Each rule dict:
#    direction : "pri_to_jit" | "sec_to_jit" | "jit_to_pri" | "jit_to_sec"
#                "merge_to_jit" (uses src_pri + src_sec + fn(a,b))
#    src / dst : dotted attr path   e.g. "payload.hf_value_a"
#    fn        : callable or None
#    note      : description shown in log
# ══════════════════════════════════════════════════════════════════════════════
HF: dict[str, dict[str, list[dict]]] = {

    "Proc1": {
        "on_entry": [
            {"direction": "pri_to_jit",
             "src": "payload.hf_value_a", "dst": "payload.hf_value_a",
             "fn": None, "note": "PRI hf_value_a(2) → JIT"},
            {"direction": "sec_to_jit",
             "src": "payload.hf_value_b", "dst": "payload.hf_value_b",
             "fn": None, "note": "SEC hf_value_b(3) → JIT"},
            # TODO: add on_entry rules
            # {"direction":"pri_to_jit","src":"payload.hf_value_a","dst":"payload.hf_value_a","fn":None,"note":"..."},
        ],
        "on_exit": [
            {"direction": "merge_to_jit",
             "src_pri": "payload.hf_value_a", "src_sec": "payload.hf_value_b",
             "dst": "payload.hf_result_p1",
             "fn": TandemCalc.multiply,
             "note": "TandemCalc.multiply  PRI(2) × SEC(3) → hf_result_p1=6"},
            {"direction": "jit_to_pri",
             "src": "payload.hf_result_p1", "dst": "payload.hf_result_p1",
             "fn": lambda v: v, "note": "JIT hf_result_p1 → PRI"},
            {"direction": "jit_to_sec",
             "src": "payload.hf_result_p1", "dst": "payload.hf_result_p1",
             "fn": lambda v: v, "note": "JIT hf_result_p1 → SEC"},
            # TODO: add gap calculation:
            # {"direction":"merge_to_jit","src_pri":"payload.hf_value_a","src_sec":"payload.hf_value_b",
            #  "dst":"payload.hf_gap_p1","fn":TandemCalc.gap,"note":"gap(2,3)=1"},
        ],
        "transition": [
            {"direction": "pri_to_jit",
             "src": "payload.hf_result_p1", "dst": "payload.transit_p1_p2",
             "fn": lambda v: f"TRANSIT_P1→P2::hf={v}",
             "note": "PRI echoes hf_result_p1 in P1→P2 corridor"},
            # TODO: add scan offset mean in corridor:
            # {"direction":"merge_to_jit","src_pri":"payload.inbound_offset_min",
            #  "src_sec":"payload.inbound_offset_min","dst":"payload.mean_inbound",
            #  "fn":TandemCalc.mean,"note":"mean inbound offset"},
        ],
    },

    "Proc2": {
        "on_entry": [
            {"direction": "pri_to_jit",
             "src": "payload.hf_result_p1", "dst": "payload.hf_result_p1",
             "fn": None, "note": "PRI carries hf_result_p1(6) → JIT"},
            {"direction": "sec_to_jit",
             "src": "payload.hf_value_c", "dst": "payload.hf_value_c",
             "fn": None, "note": "SEC hf_value_c(4) → JIT"},
        ],
        "on_exit": [
            {"direction": "merge_to_jit",
             "src_pri": "payload.hf_result_p1", "src_sec": "payload.hf_value_c",
             "dst": "payload.hf_result_p2",
             "fn": TandemCalc.multiply,
             "note": "TandemCalc.multiply  PRI(6) × SEC(4) → hf_result_p2=24"},
            {"direction": "jit_to_pri",
             "src": "payload.hf_result_p2", "dst": "payload.hf_result_p2",
             "fn": lambda v: v, "note": "JIT hf_result_p2 → PRI"},
            {"direction": "jit_to_sec",
             "src": "payload.hf_result_p2", "dst": "payload.hf_result_p2",
             "fn": lambda v: v, "note": "JIT hf_result_p2 → SEC"},
            # TODO: belt divergence ratio:
            # {"direction":"merge_to_jit","src_pri":"payload.belt_offset_min",
            #  "src_sec":"payload.belt_offset_min","dst":"payload.belt_ratio",
            #  "fn":TandemCalc.ratio,"note":"ratio Belt A / Belt B"},
        ],
        "transition": [
            {"direction": "pri_to_jit",
             "src": "payload.hf_result_p2", "dst": "payload.transit_p2_p3",
             "fn": lambda v: f"TRANSIT_P2→P3::hf={v}",
             "note": "PRI echoes hf_result_p2 in P2→P3 corridor"},
            # TODO: weighted belt offset:
            # {"direction":"merge_to_jit","src_pri":"payload.belt_offset_min",
            #  "src_sec":"payload.belt_offset_min","dst":"payload.belt_weighted",
            #  "fn":TandemCalc.weighted_sum,"note":"weighted belt offset"},
        ],
    },

    "Proc3": {
        "on_entry": [
            {"direction": "pri_to_jit",
             "src": "payload.hf_result_p2", "dst": "payload.hf_result_p2",
             "fn": None, "note": "PRI carries hf_result_p2(24) → JIT"},
            {"direction": "sec_to_jit",
             "src": "payload.hf_value_d", "dst": "payload.hf_value_d",
             "fn": None, "note": "SEC hf_value_d(2) → JIT"},
        ],
        "on_exit": [
            {"direction": "merge_to_jit",
             "src_pri": "payload.hf_result_p2", "src_sec": "payload.hf_value_d",
             "dst": "payload.hf_result_p3",
             "fn": TandemCalc.multiply,
             "note": "TandemCalc.multiply  PRI(24) × SEC(2) → hf_result_p3=48"},
            {"direction": "jit_to_pri",
             "src": "payload.hf_result_p3", "dst": "payload.hf_result_p3",
             "fn": lambda v: v, "note": "JIT hf_result_p3 → PRI"},
            {"direction": "jit_to_sec",
             "src": "payload.hf_result_p3", "dst": "payload.hf_result_p3",
             "fn": lambda v: v, "note": "JIT hf_result_p3 → SEC"},
            # TODO: convergence gap:
            # {"direction":"merge_to_jit","src_pri":"payload.conv_offset_min",
            #  "src_sec":"payload.conv_offset_min","dst":"payload.conv_gap_min",
            #  "fn":TandemCalc.gap,"note":"convergence window gap"},
            # TODO: convergence mean:
            # {"direction":"merge_to_jit","src_pri":"payload.conv_offset_min",
            #  "src_sec":"payload.conv_offset_min","dst":"payload.conv_mean_min",
            #  "fn":TandemCalc.mean,"note":"mean convergence offset"},
        ],
        "transition": [
            {"direction": "pri_to_jit",
             "src": "payload.hf_result_p3", "dst": "payload.transit_p3_p4",
             "fn": lambda v: f"TRANSIT_P3→P4::hf={v}",
             "note": "PRI echoes hf_result_p3 in P3→P4 corridor"},
        ],
    },

    "Proc4": {
        "on_entry": [
            {"direction": "pri_to_jit",
             "src": "payload.hf_result_p3", "dst": "payload.final_pri_value",
             "fn": lambda v: f"PRI_FINAL::{v}", "note": "PRI final hf_result_p3 → JIT"},
            {"direction": "sec_to_jit",
             "src": "payload.hf_result_p3", "dst": "payload.final_sec_value",
             "fn": lambda v: f"SEC_FINAL::{v}", "note": "SEC final hf_result_p3 → JIT"},
        ],
        "on_exit": [
            {"direction": "merge_to_jit",
             "src_pri": "payload.final_pri_value", "src_sec": "payload.final_sec_value",
             "dst": "payload.final_tandem_lock",
             "fn": TandemCalc.lock_seal,
             "note": "TandemCalc.lock_seal  PRI_FINAL || SEC_FINAL → final_tandem_lock"},
            {"direction": "jit_to_pri",
             "src": "payload.final_tandem_lock", "dst": "payload.final_tandem_lock",
             "fn": lambda v: f"LOCK_ACK_PRI::{v}", "note": "JIT lock → PRI"},
            {"direction": "jit_to_sec",
             "src": "payload.final_tandem_lock", "dst": "payload.final_tandem_lock",
             "fn": lambda v: f"LOCK_ACK_SEC::{v}", "note": "JIT lock → SEC"},
            # TODO: numeric chain ratio into lock:
            # {"direction":"merge_to_jit","src_pri":"payload.hf_result_p1",
            #  "src_sec":"payload.hf_result_p3","dst":"payload.hf_chain_ratio",
            #  "fn":TandemCalc.ratio,"note":"ratio p1/p3 in final lock"},
        ],
        "transition": [
            {"direction": "pri_to_jit",
             "src": "payload.final_tandem_lock", "dst": "payload.pre_release_confirm",
             "fn": lambda v: f"PRE_RELEASE_PRI::{v}", "note": "PRI pre-release confirmation"},
            {"direction": "sec_to_jit",
             "src": "payload.final_tandem_lock", "dst": "payload.pre_release_confirm",
             "fn": lambda v: f"PRE_RELEASE_SEC::{v}", "note": "SEC pre-release confirmation"},
            # TODO: final weighted sum of all hf results:
            # {"direction":"merge_to_jit","src_pri":"payload.hf_result_p1",
            #  "src_sec":"payload.hf_result_p3","dst":"payload.hf_weighted_final",
            #  "fn":TandemCalc.weighted_sum,"note":"weighted final hf value"},
        ],
    },
}


# ══════════════════════════════════════════════════════════════════════════════
#  HF EXCHANGE ENGINE
# ══════════════════════════════════════════════════════════════════════════════
class HFEngine:

    _DIR = {
        "pri_to_jit": ("pri","jit"), "sec_to_jit": ("sec","jit"),
        "jit_to_pri": ("jit","pri"), "jit_to_sec": ("jit","sec"),
        "pri_to_sec": ("pri","sec"), "sec_to_pri": ("sec","pri"),
    }
    _ARROW = {
        "pri_to_jit":"PRI ──▶ JIT", "sec_to_jit":"SEC ──▶ JIT",
        "jit_to_pri":"JIT ──▶ PRI", "jit_to_sec":"JIT ──▶ SEC",
        "pri_to_sec":"PRI ──▶ SEC", "sec_to_pri":"SEC ──▶ PRI",
        "merge_to_jit":"PRI + SEC ──▶ JIT",
    }
    _MOMENT = {"on_entry":"ON ENTRY","on_exit":"ON EXIT","transition":"TRANSITION"}

    @staticmethod
    def _get(obj, path):
        for p in path.split("."):
            obj = getattr(obj, p, None)
            if obj is None: return None
        return obj

    @staticmethod
    def _set(obj, path, value):
        parts = path.split(".")
        for p in parts[:-1]:
            obj = getattr(obj, p, None)
            if obj is None: return
        setattr(obj, parts[-1], value)

    @staticmethod
    def fire(proc_key, moment, jit, pri, sec, tel, logger=None):
        rules = HF.get(proc_key, {}).get(moment, [])
        if not rules: return

        def out(l=""):
            print(l)
            if logger: logger.info(l)

        out(f"│  ╔{'═'*62}")
        out(f"│  ║  ◈ HF EXCHANGE  {proc_key}  ·  {HFEngine._MOMENT.get(moment, moment)}")
        out(f"│  ╠{'═'*62}")

        for r in rules:
            direction = r["direction"]
            if direction == "merge_to_jit":
                val_pri = HFEngine._get(pri, r["src_pri"])
                val_sec = HFEngine._get(sec, r["src_sec"])
                result  = r["fn"](val_pri, val_sec) if r.get("fn") else (val_pri, val_sec)
                HFEngine._set(jit, r["dst"], result)
                out(f"│  ║  {HFEngine._ARROW['merge_to_jit']}")
                out(f"│  ║    note     : {r['note']}")
                out(f"│  ║    PRI src  : {r['src_pri']} = {val_pri}")
                out(f"│  ║    SEC src  : {r['src_sec']} = {val_sec}")
                out(f"│  ║    fn       : {r.get('fn')}")
                out(f"│  ║    result   : {r['dst']} = {result}")
                tel.record_hf(proc_key, moment, "merge_to_jit", r["dst"], result)
            else:
                objs = {"pri":pri,"sec":sec,"jit":jit}
                sk, dk = HFEngine._DIR.get(direction, ("",""))
                src_obj, dst_obj = objs.get(sk), objs.get(dk)
                if not src_obj or not dst_obj: continue
                raw = HFEngine._get(src_obj, r["src"])
                val = r["fn"](raw) if (r.get("fn") and raw is not None) else raw
                HFEngine._set(dst_obj, r["dst"], val)
                out(f"│  ║  {HFEngine._ARROW.get(direction, direction)}")
                out(f"│  ║    note     : {r['note']}")
                out(f"│  ║    src      : {r['src']} = {raw}")
                out(f"│  ║    dst      : {r['dst']} = {val}")
                tel.record_hf(proc_key, moment, direction, r["dst"], val)
            out(f"│  ║")

        out(f"│  ╚{'═'*62}")


# ══════════════════════════════════════════════════════════════════════════════
#  DATA MODELS
# ══════════════════════════════════════════════════════════════════════════════

# ── HF Numeric Property Space ─────────────────────────────────────────────────
@dataclass
class HFPayload:
    hf_value_a          : Optional[float] = None   # PRI Proc1 input  (2)
    hf_value_b          : Optional[float] = None   # SEC Proc1 input  (3)
    hf_value_c          : Optional[float] = None   # SEC Proc2 input  (4)
    hf_value_d          : Optional[float] = None   # SEC Proc3 input  (2)
    hf_result_p1        : Optional[float] = None   # 2×3 = 6
    hf_result_p2        : Optional[float] = None   # 6×4 = 24
    hf_result_p3        : Optional[float] = None   # 24×2 = 48
    final_pri_value     : Optional[Any]   = None
    final_sec_value     : Optional[Any]   = None
    final_tandem_lock   : Optional[Any]   = None
    pre_release_confirm : Optional[Any]   = None
    transit_p1_p2       : Optional[Any]   = None
    transit_p2_p3       : Optional[Any]   = None
    transit_p3_p4       : Optional[Any]   = None
    # TODO: add extra numeric fields for additional TandemCalc operations


# ── Package Scan Feature Space ────────────────────────────────────────────────
# These are ADDITIONAL FEATURES loaded onto each probe independently.
# NOT merged by HF engine — carried for later analysis.
@dataclass
class ScanFeatures:
    package_id           : Optional[str]   = None
    inbound_scan_id      : Optional[str]   = None
    inbound_station      : Optional[str]   = None
    inbound_timestamp    : Optional[str]   = None
    inbound_offset_min   : Optional[float] = None
    initial_scan_id      : Optional[str]   = None
    initial_station      : Optional[str]   = None
    initial_timestamp    : Optional[str]   = None
    initial_offset_min   : Optional[float] = None
    belt_scan_id         : Optional[str]   = None
    belt_station         : Optional[str]   = None
    belt_timestamp       : Optional[str]   = None
    belt_offset_min      : Optional[float] = None
    conv_scan_id         : Optional[str]   = None
    conv_station         : Optional[str]   = None
    conv_timestamp       : Optional[str]   = None
    conv_offset_min      : Optional[float] = None
    outbound_scan_id     : Optional[str]   = None
    outbound_station     : Optional[str]   = None
    outbound_timestamp   : Optional[str]   = None
    outbound_offset_min  : Optional[float] = None
    # TODO: add extra scan fields (weight, dimensions, carrier_code, etc.)


# ── Unified Payload ───────────────────────────────────────────────────────────
@dataclass
class PayloadSection:
    hf   : HFPayload    = field(default_factory=HFPayload)
    scan : ScanFeatures = field(default_factory=ScanFeatures)

    def __getattr__(self, name):
        hf_fields   = HFPayload.__dataclass_fields__
        scan_fields = ScanFeatures.__dataclass_fields__
        if name in hf_fields:   return getattr(self.hf, name)
        if name in scan_fields: return getattr(self.scan, name)
        raise AttributeError(f"PayloadSection has no attribute '{name}'")

    def __setattr__(self, name, value):
        if name in ("hf", "scan"):
            super().__setattr__(name, value); return
        try:
            hf_fields   = HFPayload.__dataclass_fields__
            scan_fields = ScanFeatures.__dataclass_fields__
        except Exception:
            super().__setattr__(name, value); return
        if name in hf_fields:        setattr(self.hf,   name, value)
        elif name in scan_fields:    setattr(self.scan, name, value)
        else:                        super().__setattr__(name, value)


@dataclass
class CallerSection:
    request_label : Optional[str] = None
    entry_marker  : Optional[str] = None
    uid           : Optional[str] = None


@dataclass
class Procedure1Section:
    phase_entry_uid       : Optional[str]               = None
    phase_exit_uid        : Optional[str]               = None
    entry_marker          : Optional[str]               = None
    timezone_object       : Optional[object]            = None
    raw_utc_datetime      : Optional[datetime.datetime] = None
    procedure_received    : Optional[bool]              = None
    merged_identity_label : Optional[str]               = None
    magnitude_trajectory  : Optional[float]             = None

@dataclass
class Procedure2Section:
    phase_entry_uid       : Optional[str]   = None
    phase_exit_uid        : Optional[str]   = None
    formatted_gmt_stamp   : Optional[str]   = None
    formatted_local_stamp : Optional[str]   = None
    acquired_utc_str      : Optional[str]   = None
    merged_time_zones     : Optional[str]   = None
    magnitude_trajectory  : Optional[float] = None

@dataclass
class Procedure3Section:
    phase_entry_uid       : Optional[str]   = None
    phase_exit_uid        : Optional[str]   = None
    display_confirmed     : Optional[bool]  = None
    display_marker        : Optional[str]   = None
    cross_stamp_verify    : Optional[str]   = None
    magnitude_trajectory  : Optional[float] = None

@dataclass
class Procedure4Section:
    phase_entry_uid       : Optional[str]   = None
    phase_exit_uid        : Optional[str]   = None
    log_confirmed         : Optional[bool]  = None
    final_tandem_lock     : Optional[str]   = None
    exit_marker           : Optional[str]   = None
    magnitude_trajectory  : Optional[float] = None


class JITPipelineArchitecture:

    class Core:
        @staticmethod
        def ms_now():
            now = datetime.datetime.now()
            return now.strftime("%Y-%m-%d  %H:%M:%S.") + f"{now.microsecond//1000:03d}"

        @staticmethod
        def obj_uid(obj):
            addr = hex(ctypes.addressof(ctypes.c_int.from_address(id(obj))))
            refs = ctypes.c_long.from_address(id(obj)).value
            return f"OBJ#{addr}@{refs}"

        @staticmethod
        def stamp(obj, event=""):
            return f"[ {JITPipelineArchitecture.Core.ms_now()} ]  {JITPipelineArchitecture.Core.obj_uid(obj)}  {event}".rstrip()

    class Data:
        class Models:

            @dataclass
            class PrimaryProbeObject:
                caller    : CallerSection    = field(default_factory=CallerSection)
                payload   : PayloadSection   = field(default_factory=PayloadSection)
                proc1     : Optional[Procedure1Section] = None
                proc2     : Optional[Procedure2Section] = None
                proc3     : Optional[Procedure3Section] = None
                proc4     : Optional[Procedure4Section] = None
                telemetry : Optional[Any]    = None
                uid       : Optional[str]    = None

            @dataclass
            class SecondaryProbeObject:
                caller    : CallerSection    = field(default_factory=CallerSection)
                payload   : PayloadSection   = field(default_factory=PayloadSection)
                proc1     : Optional[Procedure1Section] = None
                proc2     : Optional[Procedure2Section] = None
                proc3     : Optional[Procedure3Section] = None
                proc4     : Optional[Procedure4Section] = None
                uid       : Optional[str]    = None

            @dataclass
            class JITCompileObject:
                payload   : PayloadSection   = field(default_factory=PayloadSection)
                proc1     : Optional[Procedure1Section] = None
                proc2     : Optional[Procedure2Section] = None
                proc3     : Optional[Procedure3Section] = None
                proc4     : Optional[Procedure4Section] = None
                uid       : Optional[str]    = None

        class Telemetry:
            @dataclass
            class PipelineTelemetry:
                pipeline_start_ns       : Optional[int] = None
                pipeline_uid            : Optional[str] = None
                last_phase_exit_ns      : Optional[int] = None
                phase_entry_ns          : dict = field(default_factory=dict)
                phase_exit_ns           : dict = field(default_factory=dict)
                phase_duration_ns       : dict = field(default_factory=dict)
                phase_transitions_ns    : dict = field(default_factory=dict)
                phase_entry_uids        : dict = field(default_factory=dict)
                phase_exit_uids         : dict = field(default_factory=dict)
                phase_features_jit      : dict = field(default_factory=dict)
                phase_features_pri      : dict = field(default_factory=dict)
                phase_features_sec      : dict = field(default_factory=dict)
                phase_trajectory_jit    : dict = field(default_factory=dict)
                phase_trajectory_pri    : dict = field(default_factory=dict)
                phase_trajectory_sec    : dict = field(default_factory=dict)
                phase_scan_features_pri : dict = field(default_factory=dict)
                phase_scan_features_sec : dict = field(default_factory=dict)
                phase_hf_result_pri     : dict = field(default_factory=dict)
                phase_hf_result_sec     : dict = field(default_factory=dict)
                hf_log_enabled          : bool = True
                hf_log                  : list = field(default_factory=list)
                feature_log             : list = field(default_factory=list)
                phase_scan_offset_pri   : dict = field(default_factory=dict)
                phase_scan_offset_sec   : dict = field(default_factory=dict)

                def phase_enter(self, name):
                    now = time.perf_counter_ns()
                    uid = UIDGen.next(f"ENTER-{name}")
                    self.phase_entry_ns[name]   = now
                    self.phase_entry_uids[name] = uid
                    if self.last_phase_exit_ns is not None:
                        self.phase_transitions_ns[f"→{name}"] = now - self.last_phase_exit_ns
                    return uid

                def phase_exit(self, name):
                    now = time.perf_counter_ns()
                    uid = UIDGen.next(f"EXIT-{name}")
                    self.phase_exit_ns[name]     = now
                    self.phase_exit_uids[name]   = uid
                    self.phase_duration_ns[name] = now - self.phase_entry_ns.get(name, 0)
                    self.last_phase_exit_ns      = now
                    return uid

                def record_hf(self, phase, moment, direction, dst, value):
                    if self.hf_log_enabled:
                        self.hf_log.append((
                            JITPipelineArchitecture.Core.ms_now(),
                            phase, moment, direction, dst, str(value)[:120]
                        ))

                def record_tri_metrics(self, ph, fj, fp, fs, tj, tp, ts):
                    self.phase_features_jit[ph]=fj; self.phase_features_pri[ph]=fp
                    self.phase_features_sec[ph]=fs; self.phase_trajectory_jit[ph]=tj
                    self.phase_trajectory_pri[ph]=tp; self.phase_trajectory_sec[ph]=ts

                def ms_duration(self, name):
                    return f"{self.phase_duration_ns.get(name,0)/1_000_000:.3f} ms"

                def record(self, step, feature, target, value):
                    self.feature_log.append((
                        JITPipelineArchitecture.Core.ms_now(), step, feature, target, str(value)[:100]
                    ))

    class Observability: pass
    class Execution:
        class Procedures: pass
        class Orchestrator: pass

In [ ]:
# CELL 2: OBSERVABILITY, CARD LOGS, ANIMATED 3D VISUALIZATION
class JITPipelineObservability:

    class FiniteLogger:
        @staticmethod
        def trace(step_id, description, jit, pri, sec, tel):
            now    = time.perf_counter_ns()
            offset = (now - tel.pipeline_start_ns) / 1_000_000 if tel.pipeline_start_ns else 0
            jm = sys.getsizeof(jit) if jit else 0
            print(f"    ├─ [TRACE] {step_id:<5} | +{offset:09.3f}ms "
                  f"| JIT={jm}b | PRI={sys.getsizeof(pri)}b "
                  f"| SEC={sys.getsizeof(sec)}b | {description}")

    class Renderers:
        @staticmethod
        def _L(logger, line=""):
            print(line)
            if logger: logger.info(line)

        @staticmethod
        def _section_fields(section):
            return [(k, str(v)) for k, v in section.__dict__.items() if v is not None]

        @staticmethod
        def render_telemetry_summary(logger, tel):
            R = JITPipelineObservability.Renderers
            total = time.perf_counter_ns() - (tel.pipeline_start_ns or 0)
            R._L(logger, "\n  ┌─────────────────────────────────────────────────────────────")
            R._L(logger,  "  │  FINITE TRI-OBJECT TELEMETRY SUMMARY")
            R._L(logger, f"  │  Pipeline UID : {tel.pipeline_uid}")
            R._L(logger, f"  │  Wall Time    : {total/1_000_000:.3f} ms")
            for name, ns in tel.phase_duration_ns.items():
                R._L(logger, f"  │    {name:<32} {ns/1_000_000:.3f} ms")
                R._L(logger, f"  │      ENTRY : {tel.phase_entry_uids.get(name,'?')}")
                R._L(logger, f"  │      EXIT  : {tel.phase_exit_uids.get(name,'?')}")
            R._L(logger, "  └─────────────────────────────────────────────────────────────\n")

    class Console:
        @staticmethod
        def _procedure_header(number, title, entry_uid):
            print(f"\n┌────────────────────────────────────────────────────────────────")
            print(f"│  PROCEDURE {number}  ·  {title}")
            print(f"│  ENTRY UID  : {entry_uid}")
            print(f"│  {JITPipelineArchitecture.Core.ms_now()}\n│")

        @staticmethod
        def _procedure_footer(number, tel, phase_key, exit_uid):
            print(f"│  EXIT UID   : {exit_uid}")
            print(f"│  Duration   : {tel.ms_duration(phase_key)}")
            print(f"└────────────────────────────────────────────────────────────────\n")

        @staticmethod
        def _step(number, label):
            print(f"│  Step {number}  ·  {label}")

        @staticmethod
        def _field(name, value, indent=2):
            print(f"│{'    '*indent}{name:<34}  {value}")

        @staticmethod
        def _tri_probe(jit, pri, sec, label):
            C = JITPipelineArchitecture.Core
            print(f"│\n│  [ PROBE INSPECTION ]  {label}\n│  {C.ms_now()}")
            print(f"│    JIT  {C.obj_uid(jit)}  uid={getattr(jit,'uid','?')}")
            print(f"│    PRI  {C.obj_uid(pri)}  uid={getattr(pri,'uid','?')}")
            print(f"│    SEC  {C.obj_uid(sec)}  uid={getattr(sec,'uid','?')}\n│")

        @staticmethod
        def _tri_brobe(jit, pri, sec, label):
            C = JITPipelineArchitecture.Core
            print(f"│\n│  [ BROBE VERIFICATION ]  {label}\n│  {C.ms_now()}")
            print(f"│    JIT  {C.obj_uid(jit)}  uid={getattr(jit,'uid','?')}")
            print(f"│    PRI  {C.obj_uid(pri)}  uid={getattr(pri,'uid','?')}")
            print(f"│    SEC  {C.obj_uid(sec)}  uid={getattr(sec,'uid','?')}\n│")

        @staticmethod
        def _inter_procedure(jit, pri, sec, leaving, entering):
            C = JITPipelineArchitecture.Core
            print(f"\n  ╌╌╌  exiting {leaving}  →  entering {entering}  ╌╌╌")
            print(f"  {C.ms_now()}")
            print(f"  JIT  {C.obj_uid(jit)}")
            print(f"  PRI  {C.obj_uid(pri)}")
            print(f"  SEC  {C.obj_uid(sec)}\n")

    # ══════════════════════════════════════════════════════════════════════════
    #  CHRONOLOGICAL CARD LOG
    # ══════════════════════════════════════════════════════════════════════════
    class CardLog:
        W = 70

        _STYLE = {
            "PHASE_ENTRY"  : ("╔","╗","║","╚","╝","═"),
            "PHASE_EXIT"   : ("╔","╗","║","╚","╝","═"),
            "HF_EXCHANGE"  : ("┌","┐","│","└","┘","─"),
            "SCAN_FEATURE" : ("╒","╕","│","╘","╛","═"),
            "CORRIDOR"     : ("┌","┐","│","└","┘","╌"),
            "TANDEM_LOCK"  : ("╔","╗","║","╚","╝","▓"),
            "SUMMARY"      : ("█","█","█","█","█","█"),
        }
        _LABEL = {
            "PHASE_ENTRY"  : "▶ PHASE ENTRY",
            "PHASE_EXIT"   : "◀ PHASE EXIT",
            "HF_EXCHANGE"  : "◈ HF EXCHANGE",
            "SCAN_FEATURE" : "⊕ SCAN FEATURE",
            "CORRIDOR"     : "⇢ CORRIDOR",
            "TANDEM_LOCK"  : "⚑ TANDEM LOCK",
            "SUMMARY"      : "◉ SUMMARY",
        }

        @staticmethod
        def _card(card_type, title, rows, logger=None):
            W  = JITPipelineObservability.CardLog.W
            st = JITPipelineObservability.CardLog._STYLE.get(card_type,
                 JITPipelineObservability.CardLog._STYLE["HF_EXCHANGE"])
            tl,tr,ml,bl,br,hz = st
            label = JITPipelineObservability.CardLog._LABEL.get(card_type, card_type)

            def out(l):
                print(l)
                if logger: logger.info(l)

            out(f"{tl}{hz*(W-2)}{tr}")
            out(f"{ml} {label}  {title:<{W-len(label)-4}}{ml}")
            out(f"{ml}{hz*(W-2)}{ml}")
            for row in rows:
                out(f"{ml}  {row:<{W-4}}{ml}")
            out(f"{bl}{hz*(W-2)}{br}")

        @staticmethod
        def render_all(tel, pri, sec, logger=None):
            CL = JITPipelineObservability.CardLog
            W  = CL.W

            def out(l=""):
                print(l)
                if logger: logger.info(l)

            phases = list(tel.phase_duration_ns.keys())

            PROC_LABELS = {
                "Procedure1_Acquisition"   : "Proc 1 · Inbound Dock",
                "Procedure2_Transformation": "Proc 2 · Sorting Belt",
                "Procedure3_Confirmation"  : "Proc 3 · Convergence Station",
                "Procedure4_LogAndRelease" : "Proc 4 · Outbound Dock",
            }
            PROC_SCAN_ATTRS = {
                "Procedure1_Acquisition"   : ("inbound_scan_id",  "inbound_station",  "inbound_timestamp",  "inbound_offset_min"),
                "Procedure2_Transformation": ("belt_scan_id",     "belt_station",     "belt_timestamp",     "belt_offset_min"),
                "Procedure3_Confirmation"  : ("conv_scan_id",     "conv_station",     "conv_timestamp",     "conv_offset_min"),
                "Procedure4_LogAndRelease" : ("outbound_scan_id", "outbound_station", "outbound_timestamp", "outbound_offset_min"),
            }

            events = []

            for i, phase in enumerate(phases):
                plabel   = PROC_LABELS.get(phase, phase)
                entry_uid = tel.phase_entry_uids.get(phase, "?")
                exit_uid  = tel.phase_exit_uids.get(phase, "?")
                dur_ms    = tel.phase_duration_ns.get(phase, 0) / 1_000_000

                # PHASE ENTRY card
                events.append((i, 0, "PHASE_ENTRY", plabel, [
                    f"Phase     : {phase}",
                    f"Entry UID : {entry_uid}",
                    f"Step      : {i+1} of {len(phases)}",
                ]))

                # SCAN FEATURE cards — PKG001 then PKG002
                attrs = PROC_SCAN_ATTRS.get(phase)
                if attrs:
                    id_a, st_a, ts_a, off_a = attrs
                    for pkg_name, obj in [("PKG001", pri), ("PKG002", sec)]:
                        sf  = obj.payload.scan
                        sid = getattr(sf, id_a,  None)
                        sst = getattr(sf, st_a,  None)
                        sts = getattr(sf, ts_a,  None)
                        sof = getattr(sf, off_a, None)
                        if sid:
                            events.append((i, 1, "SCAN_FEATURE",
                                           f"{pkg_name} @ {sst or '?'}", [
                                f"Package   : {pkg_name}",
                                f"Station   : {sst or '—'}",
                                f"Timestamp : {sts or '—'}",
                                f"Offset    : {sof} min",
                                f"Scan ID   : {sid}",
                            ]))

                # HF EXCHANGE cards
                phase_hf = [e for e in tel.hf_log if e[1] == phase]
                moment_groups: dict = {}
                for ts, ph, moment, direction, dst, val in phase_hf:
                    moment_groups.setdefault(moment, []).append((ts, direction, dst, val))

                for m_order, moment in enumerate(["on_entry", "on_exit", "transition"]):
                    entries = moment_groups.get(moment, [])
                    if not entries: continue
                    hf_rows = [f"Moment    : {moment}"]
                    for ts, direction, dst, val in entries:
                        hf_rows += [f"  {ts}", f"  {direction:<18}  {dst}",
                                    f"  value = {str(val)[:50]}", f"  {'·'*50}"]
                    events.append((i, 2+m_order, "HF_EXCHANGE",
                                   f"{phase} · {moment.upper()}", hf_rows))

                # CORRIDOR card
                if i < len(phases) - 1:
                    next_label = PROC_LABELS.get(phases[i+1], phases[i+1])
                    corr_rows  = [f"Exiting   : {plabel}", f"Entering  : {next_label}"]
                    trans_ns   = tel.phase_transitions_ns.get(f"→{phases[i+1]}")
                    if trans_ns:
                        corr_rows.append(f"Gap       : {trans_ns/1_000_000:.3f} ms")
                    events.append((i, 5, "CORRIDOR", f"corridor {i+1}→{i+2}", corr_rows))

                # PHASE EXIT card
                hf_vals = {
                    "Procedure1_Acquisition"   : ("hf_result_p1", pri.payload.hf_result_p1),
                    "Procedure2_Transformation": ("hf_result_p2", pri.payload.hf_result_p2),
                    "Procedure3_Confirmation"  : ("hf_result_p3", pri.payload.hf_result_p3),
                    "Procedure4_LogAndRelease" : ("final_tandem_lock", pri.payload.final_tandem_lock),
                }
                hf_key, hf_val = hf_vals.get(phase, (None, None))
                exit_rows = [
                    f"Phase     : {phase}",
                    f"Exit UID  : {exit_uid}",
                    f"Duration  : {dur_ms:.3f} ms",
                    f"PRI feats : {tel.phase_features_pri.get(phase,0)}",
                    f"SEC feats : {tel.phase_features_sec.get(phase,0)}",
                    f"JIT feats : {tel.phase_features_jit.get(phase,0)}",
                ]
                if hf_key:
                    exit_rows.append(f"HF result : {hf_key} = {hf_val}")
                events.append((i, 6, "PHASE_EXIT", plabel, exit_rows))

            # TANDEM LOCK summary card
            events.append((len(phases), 9, "TANDEM_LOCK",
                           "FINAL TANDEM LOCK — PIPELINE COMPLETE", [
                f"PKG001 lock : {str(pri.payload.final_tandem_lock)[:60]}",
                f"PKG002 lock : {str(sec.payload.final_tandem_lock)[:60]}",
                f"",
                f"HF chain    : 2×3={pri.payload.hf_result_p1}  ×4={pri.payload.hf_result_p2}  ×2={pri.payload.hf_result_p3}",
                f"",
                f"TandemCalc used : .multiply (P1,P2,P3)  ·  .lock_seal (P4)",
                f"",
                f"TODO: TandemCalc.gap / .ratio / .mean / .weighted_sum",
            ]))

            # render
            out(); out("═"*W)
            out(f"  CHRONOLOGICAL PIPELINE LOG  ·  {len(events)} cards")
            out("═"*W); out()

            for ev in sorted(events, key=lambda e: (e[0], e[1])):
                _, _, card_type, title, rows = ev
                CL._card(card_type, title, rows, logger)
                out()

            out("═"*W); out("  END OF LOG"); out("═"*W); out()

    # ══════════════════════════════════════════════════════════════════════════
    #  3D ANIMATED VISUALIZATION
    # ══════════════════════════════════════════════════════════════════════════
    class Visualizations:
        @staticmethod
        def render_animated_telemetry(tel, pri, sec):
            phases = list(tel.phase_duration_ns.keys())
            if not phases: return

            n       = len(phases)
            dur_ms  = [tel.phase_duration_ns[p]/1_000_000 for p in phases]
            fj_vals = [tel.phase_features_jit.get(p,0) for p in phases]
            fp_vals = [tel.phase_features_pri.get(p,0) for p in phases]
            fs_vals = [tel.phase_features_sec.get(p,0) for p in phases]

            cum_t  = np.cumsum(dur_ms)
            pre_t  = -cum_t[-1]*0.10
            t_pts  = np.concatenate([[pre_t], cum_t])
            fj_pts = np.concatenate([[0], np.cumsum(fj_vals)])
            fp_pts = np.concatenate([[0], np.cumsum(fp_vals)])
            fs_pts = np.concatenate([[0], np.cumsum(fs_vals)])

            LANE_JIT, LANE_PRI, LANE_SEC = 0.0, 1.0, 2.0
            n_pts    = len(t_pts)
            idx_orig = np.arange(n_pts, dtype=float)
            fps_ph   = 20
            t_smooth = np.linspace(0, n_pts-1, (n_pts-1)*fps_ph+1)
            xs   = np.interp(t_smooth, idx_orig, t_pts)
            zj_s = np.interp(t_smooth, idx_orig, fj_pts)
            zp_s = np.interp(t_smooth, idx_orig, fp_pts)
            zs_s = np.interp(t_smooth, idx_orig, fs_pts)

            # execution surface
            x_surf = np.linspace(pre_t, cum_t[-1], 40)
            y_surf = np.linspace(-0.5, 0.5, 10)
            xm, ym = np.meshgrid(x_surf, y_surf)
            z_base = np.interp(x_surf, t_pts, fj_pts)
            zm_surf = (np.outer(np.ones(len(y_surf)), z_base)
                       + np.sin(xm*0.6)*np.cos(ym*4)*0.3)

            MOMENT_COLOR  = {"on_entry":"#FFD700","on_exit":"#FF6B6B","transition":"#00E5FF"}
            MOMENT_SYMBOL = {"on_entry":"diamond","on_exit":"square","transition":"circle"}

            phase_to_t  = {ph: cum_t[i] for i, ph in enumerate(phases)}
            cum_fj = np.cumsum(fj_vals); cum_fp = np.cumsum(fp_vals); cum_fs = np.cumsum(fs_vals)
            phase_to_fj = {ph: cum_fj[i] for i, ph in enumerate(phases)}
            phase_to_fp = {ph: cum_fp[i] for i, ph in enumerate(phases)}
            phase_to_fs = {ph: cum_fs[i] for i, ph in enumerate(phases)}

            hf_events = {m: {"x":[],"y":[],"z":[],"text":[],"cx":[],"cy":[],"cz":[]}
                         for m in ("on_entry","on_exit","transition")}

            for entry in tel.hf_log:
                ts, phase, moment, direction, dst, val = entry
                if phase not in phase_to_t: continue
                bucket = moment if moment in hf_events else "on_entry"
                px = phase_to_t[phase]
                zj = phase_to_fj[phase]; zp = phase_to_fp[phase]; zs = phase_to_fs[phase]
                d  = direction
                if "pri" in d and "jit" in d:   la,za,lb,zb = LANE_JIT,zj,LANE_PRI,zp
                elif "sec" in d and "jit" in d: la,za,lb,zb = LANE_JIT,zj,LANE_SEC,zs
                else:                           la,za,lb,zb = LANE_PRI,zp,LANE_SEC,zs
                hf_events[bucket]["x"].append(px)
                hf_events[bucket]["y"].append((la+lb)/2)
                hf_events[bucket]["z"].append((za+zb)/2)
                hf_events[bucket]["text"].append(f"{phase}<br>{moment}<br>{direction}<br>{dst}<br>{str(val)[:40]}")
                hf_events[bucket]["cx"] += [px,px,None]
                hf_events[bucket]["cy"] += [la,lb,None]
                hf_events[bucket]["cz"] += [za,zb,None]

            # scan feature drop spikes — one per scan event per probe
            all_offsets = sorted([s["offset_min"] for s in PKG001_SCANS+PKG002_SCANS])
            total_real  = all_offsets[-1]-all_offsets[0]
            total_wall  = cum_t[-1]

            def scan_to_x(off):
                if total_real == 0: return 0.0
                return ((off-all_offsets[0])/total_real)*total_wall

            SCAN_PRI = [
                ("inbound_offset_min",  pri.payload.scan.inbound_scan_id,  "Inbound"),
                ("initial_offset_min",  pri.payload.scan.initial_scan_id,  "Initial"),
                ("belt_offset_min",     pri.payload.scan.belt_scan_id,     "Belt A"),
                ("conv_offset_min",     pri.payload.scan.conv_scan_id,     "Conv"),
                ("outbound_offset_min", pri.payload.scan.outbound_scan_id, "Outbound"),
            ]
            SCAN_SEC = [
                ("inbound_offset_min",  sec.payload.scan.inbound_scan_id,  "Inbound"),
                ("initial_offset_min",  sec.payload.scan.initial_scan_id,  "Initial"),
                ("belt_offset_min",     sec.payload.scan.belt_scan_id,     "Belt B"),
                ("conv_offset_min",     sec.payload.scan.conv_scan_id,     "Conv"),
                ("outbound_offset_min", sec.payload.scan.outbound_scan_id, "Outbound"),
            ]

            spike_x, spike_y, spike_z = [], [], []
            for off_a, sid, lbl in SCAN_PRI:
                off = getattr(pri.payload.scan, off_a, None)
                if off is None or sid is None: continue
                sx = scan_to_x(off)
                np_ = min(phases, key=lambda p: abs(phase_to_t[p]-sx))
                sz  = phase_to_fp[np_]
                spike_x += [sx,sx,None]; spike_y += [LANE_PRI,LANE_PRI,None]
                spike_z += [sz, sz-1.2, None]
            for off_a, sid, lbl in SCAN_SEC:
                off = getattr(sec.payload.scan, off_a, None)
                if off is None or sid is None: continue
                sx = scan_to_x(off)
                np_ = min(phases, key=lambda p: abs(phase_to_t[p]-sx))
                sz  = phase_to_fs[np_]
                spike_x += [sx,sx,None]; spike_y += [LANE_SEC,LANE_SEC,None]
                spike_z += [sz, sz-1.2, None]

            # convergence ring
            conv_x     = scan_to_x(PKG001_SCANS[3]["offset_min"])
            theta      = np.linspace(0, 2*np.pi, 60)
            conv_ring_y = LANE_JIT+1.0 + 0.6*np.cos(theta)
            conv_ring_z = max(fj_pts)*0.5 + max(fj_pts)*0.4*np.sin(theta)
            conv_ring_x = np.full_like(theta, conv_x)

            # TandemCalc labels above JIT lane
            hf_results = [
                (phase_to_t["Procedure1_Acquisition"],    fj_pts[1], "2×3=6"),
                (phase_to_t["Procedure2_Transformation"], fj_pts[2], "6×4=24"),
                (phase_to_t["Procedure3_Confirmation"],   fj_pts[3], "24×2=48"),
            ]

            # phase boundary annotations
            phase_annots = []
            for i, ph in enumerate(phases):
                short = (ph.replace("Procedure","P")
                           .replace("_Acquisition","").replace("_Transformation","")
                           .replace("_Confirmation","").replace("_LogAndRelease",""))
                phase_annots.append(dict(
                    x=cum_t[i], y=LANE_SEC+0.6, z=0,
                    text=short, showarrow=False,
                    font=dict(size=9, color="rgba(255,255,255,0.55)"), xanchor="center",
                ))

            fig = go.Figure()

            fig.add_trace(go.Surface(x=x_surf,y=y_surf,z=zm_surf,
                colorscale="Viridis",opacity=0.15,showscale=False,name="Exec Surface",hoverinfo="skip"))

            fig.add_trace(go.Scatter3d(x=t_pts,y=[LANE_JIT]*n_pts,z=fj_pts,
                mode="lines",line=dict(color="#00d2ff",width=5),name="JIT accumulation"))
            fig.add_trace(go.Scatter3d(x=t_pts,y=[LANE_PRI]*n_pts,z=fp_pts,
                mode="lines",line=dict(color="#ff007f",width=3),name="PKG001 accumulation"))
            fig.add_trace(go.Scatter3d(x=t_pts,y=[LANE_SEC]*n_pts,z=fs_pts,
                mode="lines",line=dict(color="#00ff88",width=3),name="PKG002 accumulation"))

            # scan feature spikes
            fig.add_trace(go.Scatter3d(x=spike_x,y=spike_y,z=spike_z,
                mode="lines",line=dict(color="#FFAA00",width=2),
                name="⊕ Scan Feature Added",hoverinfo="skip"))

            # convergence ring
            fig.add_trace(go.Scatter3d(x=conv_ring_x,y=conv_ring_y,z=conv_ring_z,
                mode="lines",line=dict(color="#FF00FF",width=4),name="⚡ Convergence Event"))

            # TandemCalc result labels
            for tx,tz,lbl in hf_results:
                fig.add_trace(go.Scatter3d(x=[tx],y=[LANE_JIT-0.3],z=[tz+0.5],
                    mode="text",text=[lbl],textfont=dict(size=11,color="#FFD700"),
                    showlegend=False,hoverinfo="skip"))

            # HF connector lines
            for moment in ("on_entry","on_exit","transition"):
                ev = hf_events[moment]
                if ev["cx"]:
                    fig.add_trace(go.Scatter3d(x=ev["cx"],y=ev["cy"],z=ev["cz"],
                        mode="lines",line=dict(color=MOMENT_COLOR[moment],width=3,dash="dot"),
                        name=f"HF {moment} link",hoverinfo="skip"))

            # HF markers
            for moment in ("on_entry","on_exit","transition"):
                ev = hf_events[moment]
                if ev["x"]:
                    fig.add_trace(go.Scatter3d(x=ev["x"],y=ev["y"],z=ev["z"],
                        mode="markers",
                        marker=dict(symbol=MOMENT_SYMBOL[moment],size=8,color=MOMENT_COLOR[moment],
                                    line=dict(width=1,color="white")),
                        text=ev["text"],hoverinfo="text",name=f"HF {moment}"))

            n_static = len(fig.data)

            fig.add_trace(go.Scatter3d(x=[xs[0]],y=[LANE_JIT],z=[zj_s[0]],
                mode="markers+text",marker=dict(size=14,color="#00d2ff",symbol="circle",
                line=dict(color="white",width=2)),text=["JIT"],textposition="top center",name="JIT (live)"))
            fig.add_trace(go.Scatter3d(x=[xs[0]],y=[LANE_PRI],z=[zp_s[0]],
                mode="markers+text",marker=dict(size=11,color="#ff007f",symbol="circle",
                line=dict(color="white",width=1)),text=["PKG001"],textposition="top center",name="PKG001 (live)"))
            fig.add_trace(go.Scatter3d(x=[xs[0]],y=[LANE_SEC],z=[zs_s[0]],
                mode="markers+text",marker=dict(size=11,color="#00ff88",symbol="circle",
                line=dict(color="white",width=1)),text=["PKG002"],textposition="top center",name="PKG002 (live)"))
            fig.add_trace(go.Scatter3d(x=[xs[0],xs[0]],y=[LANE_JIT,LANE_PRI],z=[zj_s[0],zp_s[0]],
                mode="lines",line=dict(color="yellow",width=4,dash="dot"),
                name="live JIT↔PKG001",hoverinfo="skip"))
            fig.add_trace(go.Scatter3d(x=[xs[0],xs[0]],y=[LANE_JIT,LANE_SEC],z=[zj_s[0],zs_s[0]],
                mode="lines",line=dict(color="orange",width=4,dash="dot"),
                name="live JIT↔PKG002",hoverinfo="skip"))

            i_jit=n_static; i_pri=n_static+1; i_sec=n_static+2
            i_lp=n_static+3; i_ls=n_static+4
            anim_idx = [i_jit,i_pri,i_sec,i_lp,i_ls]

            frames = []
            for i in range(len(t_smooth)):
                jg = np.sin(i*np.pi/3)*0.08
                frames.append(go.Frame(data=[
                    go.Scatter3d(x=[xs[i]],y=[LANE_JIT],z=[zj_s[i]]),
                    go.Scatter3d(x=[xs[i]],y=[LANE_PRI],z=[zp_s[i]]),
                    go.Scatter3d(x=[xs[i]],y=[LANE_SEC],z=[zs_s[i]]),
                    go.Scatter3d(x=[xs[i],xs[i]],y=[LANE_JIT,LANE_PRI+jg],z=[zj_s[i],zp_s[i]]),
                    go.Scatter3d(x=[xs[i],xs[i]],y=[LANE_JIT,LANE_SEC-jg],z=[zj_s[i],zs_s[i]]),
                ], name=f"F{i}", traces=anim_idx))
            fig.frames = frames

            ytick_vals = [LANE_JIT,LANE_PRI,LANE_SEC]
            ytick_text = ["JIT Compile Obj","PKG001 (Primary)","PKG002 (Secondary)"]

            fig.update_layout(
                title=dict(text=(
                    "Denver Hub Package Pipeline — Feature Accumulation & HF Exchange<br>"
                    "<sup>X=Wall Time · Y=Object Lane · Z=Feature Mass · "
                    "Gold=on_entry · Coral=on_exit · Cyan=transition · "
                    "Orange=Scan Added · Magenta=Convergence · Yellow labels=TandemCalc</sup>"),
                    font=dict(size=13)),
                scene=dict(
                    xaxis=dict(title="Wall Time (ms)",backgroundcolor="#0d1117",
                               gridcolor="#1e2d3d",showbackground=True),
                    yaxis=dict(title="Object Lane",tickvals=ytick_vals,ticktext=ytick_text,
                               backgroundcolor="#0d1117",gridcolor="#1e2d3d",
                               showbackground=True,range=[-0.6,2.8]),
                    zaxis=dict(title="Accumulated Feature Mass",backgroundcolor="#0d1117",
                               gridcolor="#1e2d3d",showbackground=True),
                    camera=dict(eye=dict(x=1.6,y=-1.8,z=1.2),up=dict(x=0,y=0,z=1)),
                    bgcolor="#0d1117",
                    annotations=phase_annots,
                ),
                autosize=False,width=1150,height=740,
                template="plotly_dark",paper_bgcolor="#0d1117",
                legend=dict(x=1.01,y=0.98,bgcolor="rgba(0,0,0,0.5)",font=dict(size=10)),
                updatemenus=[dict(type="buttons",showactive=False,y=0.02,x=0.12,
                    xanchor="right",yanchor="bottom",
                    buttons=[
                        dict(label="▶ Play",method="animate",
                             args=[None,dict(frame=dict(duration=125,redraw=True),
                                             fromcurrent=True,transition=dict(duration=80,easing="linear"))]),
                        dict(label="⏸ Pause",method="animate",
                             args=[[None],dict(frame=dict(duration=0,redraw=False),
                                               mode="immediate",transition=dict(duration=0))]),
                    ])],
                sliders=[dict(
                    steps=[dict(method="animate",
                                args=[[f"F{i}"],dict(mode="immediate",
                                    frame=dict(duration=125,redraw=True),
                                    transition=dict(duration=0))],
                                label=f"{xs[i]:.1f}ms")
                           for i in range(0,len(t_smooth),max(1,len(t_smooth)//50))],
                    x=0.0,y=0.0,len=1.0,
                    currentvalue=dict(prefix="Time: ",suffix=" ms",
                                      font=dict(size=11,color="white"),
                                      visible=True,xanchor="center"),
                    transition=dict(duration=0),pad=dict(t=50,b=10),
                )],
            )
            fig.show()


CardLog = JITPipelineObservability.CardLog
JITPipelineArchitecture.Observability = JITPipelineObservability

In [ ]:
# CELL 3: PROCEDURE 1 — Inbound Dock Arrival
class DataAcquisitionProcedure1:
    @staticmethod
    def execute(jit, pri, sec, tel):
        Obs  = JITPipelineArchitecture.Observability
        Con  = Obs.Console
        Log  = Obs.FiniteLogger
        Core = JITPipelineArchitecture.Core

        entry_uid = tel.phase_enter("Procedure1_Acquisition")
        Con._procedure_header(1, "Inbound Dock Arrival & Gap Calculation", entry_uid)

        jit.proc1 = Procedure1Section(); jit.proc1.phase_entry_uid = entry_uid
        pri.proc1 = Procedure1Section(); pri.proc1.phase_entry_uid = entry_uid
        sec.proc1 = Procedure1Section(); sec.proc1.phase_entry_uid = entry_uid

        Con._tri_probe(jit, pri, sec, "Procedure 1 ENTRY")

        HFEngine.fire("Proc1", "on_entry", jit, pri, sec, tel)
        Log.trace("1.0", "HF on_entry complete", jit, pri, sec, tel)

        Con._step("1.1", "Entry markers")
        ms = Core.ms_now()
        jit.proc1.entry_marker = f"[ {ms} ] JIT entry_uid={entry_uid}"
        pri.proc1.entry_marker = f"[ {ms} ] PKG001 entry_uid={entry_uid}"
        sec.proc1.entry_marker = f"[ {ms} ] PKG002 entry_uid={entry_uid}"

        Con._step("1.2", "Core data — timezone + UTC snapshot")
        pri.proc1.procedure_received = True
        sec.proc1.procedure_received = True
        jit.proc1.timezone_object    = pytz.timezone("America/Denver")
        jit.proc1.raw_utc_datetime   = datetime.datetime.now(pytz.utc)
        # TODO: acquire additional data here
        # example — stamp raw UTC onto PKG001 proc1:
        # pri.proc1.raw_utc_datetime = jit.proc1.raw_utc_datetime

        Con._step("1.3", "Identity merge — package labels into JIT")
        jit.proc1.merged_identity_label = (
            f"MERGED::[{pri.caller.request_label}:inbound={pri.payload.scan.inbound_offset_min}min]"
            f"[{sec.caller.request_label}:inbound={sec.payload.scan.inbound_offset_min}min]"
            f"[entry={entry_uid}]"
        )
        Con._field("merged_identity_label", jit.proc1.merged_identity_label)
        Log.trace("1.3", "Identity merged", jit, pri, sec, tel)

        fj = len(Obs.Renderers._section_fields(jit.proc1))
        fp = len(Obs.Renderers._section_fields(pri.proc1))
        fs = len(Obs.Renderers._section_fields(sec.proc1))
        ex = max(1, time.perf_counter_ns() - tel.phase_entry_ns["Procedure1_Acquisition"]) / 1_000_000
        tj = (ex*fj)/max(1,sum(ord(c) for c in Core.obj_uid(jit))+fj)
        tp = (ex*fp)/max(1,sum(ord(c) for c in Core.obj_uid(pri))+fp)
        ts = (ex*fs)/max(1,sum(ord(c) for c in Core.obj_uid(sec))+fs)
        jit.proc1.magnitude_trajectory = tj
        pri.proc1.magnitude_trajectory = tp
        sec.proc1.magnitude_trajectory = ts
        tel.record_tri_metrics("Procedure1_Acquisition", fj, fp, fs, tj, tp, ts)

        HFEngine.fire("Proc1", "on_exit", jit, pri, sec, tel)
        Log.trace("1.E", f"HF on_exit complete  hf_result_p1={jit.payload.hf_result_p1}", jit, pri, sec, tel)
        # TODO: manipulate hf_result_p1 after exit
        # example: pri.payload.same_wave = (jit.payload.hf_result_p1 or 0) <= 6

        exit_uid = tel.phase_exit("Procedure1_Acquisition")
        jit.proc1.phase_exit_uid = exit_uid
        pri.proc1.phase_exit_uid = exit_uid
        sec.proc1.phase_exit_uid = exit_uid

        Con._tri_brobe(jit, pri, sec, "Procedure 1 EXIT")
        Con._procedure_footer(1, tel, "Procedure1_Acquisition", exit_uid)
        return jit, pri, sec

JITPipelineArchitecture.Execution.Procedures.Proc1 = DataAcquisitionProcedure1

In [ ]:
# CELL 4: PROCEDURE 2 — Sorting Belt Divergence
class DataAcquisitionProcedure2:
    @staticmethod
    def execute(jit, pri, sec, tel):
        Obs  = JITPipelineArchitecture.Observability
        Con  = Obs.Console
        Log  = Obs.FiniteLogger
        Core = JITPipelineArchitecture.Core

        entry_uid = tel.phase_enter("Procedure2_Transformation")
        Con._procedure_header(2, "Sorting Belt Divergence — Belt A vs Belt B", entry_uid)

        jit.proc2 = Procedure2Section(); jit.proc2.phase_entry_uid = entry_uid
        pri.proc2 = Procedure2Section(); pri.proc2.phase_entry_uid = entry_uid
        sec.proc2 = Procedure2Section(); sec.proc2.phase_entry_uid = entry_uid

        Con._tri_probe(jit, pri, sec, "Procedure 2 ENTRY")

        HFEngine.fire("Proc2", "on_entry", jit, pri, sec, tel)
        Log.trace("2.0", "HF on_entry complete — belt offsets in JIT", jit, pri, sec, tel)

        Con._step("2.1", "Format scan timestamps")
        jit.proc2.formatted_gmt_stamp   = jit.proc1.raw_utc_datetime.strftime("%H:%M:%S GMT")
        jit.proc2.formatted_local_stamp = f"LOCAL_{jit.proc1.raw_utc_datetime.strftime('%S.%f')}"
        # TODO: add more timestamp formats
        # example: pri.proc2.formatted_gmt_stamp = jit.proc1.raw_utc_datetime.isoformat()

        Con._step("2.2", "Belt scan UTC strings")
        pri.proc2.acquired_utc_str = (
            f"PKG001_{pri.payload.scan.belt_station}_UTC_{jit.proc1.raw_utc_datetime.microsecond}"
        )
        sec.proc2.acquired_utc_str = (
            f"PKG002_{sec.payload.scan.belt_station}_UTC_{jit.proc1.raw_utc_datetime.microsecond + 500}"
        )

        Con._step("2.3", "Routing merge — belt divergence")
        jit.proc2.merged_time_zones = (
            f"BELT_SYNC::[{pri.proc2.acquired_utc_str}]<=>[{sec.proc2.acquired_utc_str}]"
            f"[entry={entry_uid}]"
        )
        Con._field("merged_time_zones", jit.proc2.merged_time_zones)
        Log.trace("2.3", "Belt routing merged", jit, pri, sec, tel)

        fj = len(Obs.Renderers._section_fields(jit.proc2))
        fp = len(Obs.Renderers._section_fields(pri.proc2))
        fs = len(Obs.Renderers._section_fields(sec.proc2))
        ex = max(1, time.perf_counter_ns() - tel.phase_entry_ns["Procedure2_Transformation"]) / 1_000_000
        tj = (ex*fj)/max(1,sum(ord(c) for c in Core.obj_uid(jit))+fj)
        tp = (ex*fp)/max(1,sum(ord(c) for c in Core.obj_uid(pri))+fp)
        ts = (ex*fs)/max(1,sum(ord(c) for c in Core.obj_uid(sec))+fs)
        jit.proc2.magnitude_trajectory = tj
        pri.proc2.magnitude_trajectory = tp
        sec.proc2.magnitude_trajectory = ts
        tel.record_tri_metrics("Procedure2_Transformation", fj, fp, fs, tj, tp, ts)

        HFEngine.fire("Proc2", "on_exit", jit, pri, sec, tel)
        Log.trace("2.E", f"HF on_exit complete  hf_result_p2={jit.payload.hf_result_p2}", jit, pri, sec, tel)
        # TODO: manipulate hf_result_p2 after exit
        # example: jit.payload.expected_conv_wait = round((jit.payload.hf_result_p2 or 0) * 0.5, 1)

        exit_uid = tel.phase_exit("Procedure2_Transformation")
        jit.proc2.phase_exit_uid = exit_uid
        pri.proc2.phase_exit_uid = exit_uid
        sec.proc2.phase_exit_uid = exit_uid

        Con._tri_brobe(jit, pri, sec, "Procedure 2 EXIT")
        Con._procedure_footer(2, tel, "Procedure2_Transformation", exit_uid)
        return jit, pri, sec

JITPipelineArchitecture.Execution.Procedures.Proc2 = DataAcquisitionProcedure2

In [ ]:
# CELL 5: PROCEDURE 3 — Convergence Station Rendezvous
class DataAcquisitionProcedure3:
    @staticmethod
    def execute(jit, pri, sec, tel):
        Obs  = JITPipelineArchitecture.Observability
        Con  = Obs.Console
        Log  = Obs.FiniteLogger
        Core = JITPipelineArchitecture.Core

        entry_uid = tel.phase_enter("Procedure3_Confirmation")
        Con._procedure_header(3, "Convergence Station — Physical Package Rendezvous", entry_uid)

        jit.proc3 = Procedure3Section(); jit.proc3.phase_entry_uid = entry_uid
        pri.proc3 = Procedure3Section(); pri.proc3.phase_entry_uid = entry_uid
        sec.proc3 = Procedure3Section(); sec.proc3.phase_entry_uid = entry_uid

        Con._tri_probe(jit, pri, sec, "Procedure 3 ENTRY")

        HFEngine.fire("Proc3", "on_entry", jit, pri, sec, tel)
        Log.trace("3.0", "HF on_entry complete — convergence offsets in JIT", jit, pri, sec, tel)

        Con._step("3.1", "Convergence confirmation markers")
        ms = Core.ms_now()
        jit.proc3.display_confirmed = True
        pri.proc3.display_confirmed = True
        sec.proc3.display_confirmed = True
        pri.proc3.display_marker = (
            f"PKG001_CONV_OK[t={pri.payload.scan.conv_offset_min}min]"
            f"[scan={pri.payload.scan.conv_scan_id[:8]}...][{ms}]"
        )
        sec.proc3.display_marker = (
            f"PKG002_CONV_OK[t={sec.payload.scan.conv_offset_min}min]"
            f"[scan={sec.payload.scan.conv_scan_id[:8]}...][{ms}]"
        )
        # TODO: add convergence confirmation steps
        # example: pri.proc3.display_marker += f"[hf_chain={pri.payload.hf_result_p2}]"

        Con._step("3.2", "Cross-verification merge — both packages at Convergence")
        jit.proc3.cross_stamp_verify = (
            f"CONVERGENCE_VERIFIED::({pri.proc3.display_marker})&({sec.proc3.display_marker})"
            f"[entry={entry_uid}]"
        )
        Con._field("cross_stamp_verify", jit.proc3.cross_stamp_verify[:80]+"...")
        Log.trace("3.2", "Convergence cross-verify merged", jit, pri, sec, tel)
        # TODO: manipulate cross-verification
        # example: jit.proc3.cross_stamp_verify += f"[hf_p3={jit.payload.hf_result_p3}]"

        fj = len(Obs.Renderers._section_fields(jit.proc3))
        fp = len(Obs.Renderers._section_fields(pri.proc3))
        fs = len(Obs.Renderers._section_fields(sec.proc3))
        ex = max(1, time.perf_counter_ns() - tel.phase_entry_ns["Procedure3_Confirmation"]) / 1_000_000
        tj = (ex*fj)/max(1,sum(ord(c) for c in Core.obj_uid(jit))+fj)
        tp = (ex*fp)/max(1,sum(ord(c) for c in Core.obj_uid(pri))+fp)
        ts = (ex*fs)/max(1,sum(ord(c) for c in Core.obj_uid(sec))+fs)
        jit.proc3.magnitude_trajectory = tj
        pri.proc3.magnitude_trajectory = tp
        sec.proc3.magnitude_trajectory = ts
        tel.record_tri_metrics("Procedure3_Confirmation", fj, fp, fs, tj, tp, ts)

        HFEngine.fire("Proc3", "on_exit", jit, pri, sec, tel)
        Log.trace("3.E", f"HF on_exit complete  hf_result_p3={jit.payload.hf_result_p3}", jit, pri, sec, tel)
        # TODO: manipulate hf_result_p3 after exit
        # example: jit.payload.conv_window_min = abs(
        #     (sec.payload.scan.conv_offset_min or 0) - (pri.payload.scan.conv_offset_min or 0))

        exit_uid = tel.phase_exit("Procedure3_Confirmation")
        jit.proc3.phase_exit_uid = exit_uid
        pri.proc3.phase_exit_uid = exit_uid
        sec.proc3.phase_exit_uid = exit_uid

        Con._tri_brobe(jit, pri, sec, "Procedure 3 EXIT")
        Con._procedure_footer(3, tel, "Procedure3_Confirmation", exit_uid)
        return jit, pri, sec

JITPipelineArchitecture.Execution.Procedures.Proc3 = DataAcquisitionProcedure3

In [ ]:
# CELL 6: PROCEDURE 4 — Outbound Dock Departure & Final Lock
class DataAcquisitionProcedure4:
    @staticmethod
    def execute(jit, pri, sec, tel):
        Obs  = JITPipelineArchitecture.Observability
        Con  = Obs.Console
        Log  = Obs.FiniteLogger
        Core = JITPipelineArchitecture.Core

        entry_uid = tel.phase_enter("Procedure4_LogAndRelease")
        Con._procedure_header(4, "Outbound Dock — Final Lock & JIT Release", entry_uid)

        jit.proc4 = Procedure4Section(); jit.proc4.phase_entry_uid = entry_uid
        pri.proc4 = Procedure4Section(); pri.proc4.phase_entry_uid = entry_uid
        sec.proc4 = Procedure4Section(); sec.proc4.phase_entry_uid = entry_uid

        Con._tri_probe(jit, pri, sec, "Procedure 4 ENTRY")

        HFEngine.fire("Proc4", "on_entry", jit, pri, sec, tel)
        Log.trace("4.0", "HF on_entry complete — outbound scan IDs in JIT", jit, pri, sec, tel)

        Con._step("4.1", "Outbound log confirmations")
        pri.proc4.log_confirmed = True
        sec.proc4.log_confirmed = True
        # TODO: add log/seal steps
        # example: jit.proc4.entry_marker = (
        #     f"PIPELINE={tel.pipeline_uid}|PKG001={pri.payload.scan.outbound_offset_min}min"
        #     f"|PKG002={sec.payload.scan.outbound_offset_min}min")

        Con._step("4.2", "Final tandem lock — seal both package journeys")
        jit.proc4.final_tandem_lock = (
            f"OUTBOUND_LOCK::"
            f"PKG001(inbound={pri.payload.scan.inbound_offset_min}min"
            f"|belt={pri.payload.scan.belt_station}"
            f"|conv={pri.payload.scan.conv_offset_min}min"
            f"|outbound={pri.payload.scan.outbound_offset_min}min)"
            f"||PKG002(inbound={sec.payload.scan.inbound_offset_min}min"
            f"|belt={sec.payload.scan.belt_station}"
            f"|conv={sec.payload.scan.conv_offset_min}min"
            f"|outbound={sec.payload.scan.outbound_offset_min}min)"
            f"[hf_chain=6→24→48]"
            f"[entry={entry_uid}]"
        )
        Con._field("final_tandem_lock", jit.proc4.final_tandem_lock[:80]+"...")
        Log.trace("4.2", "Final lock written", jit, pri, sec, tel)
        # TODO: manipulate final lock
        # example: elapsed = (time.perf_counter_ns()-tel.pipeline_start_ns)/1_000_000
        # jit.proc4.final_tandem_lock += f"[pipeline_wall={elapsed:.3f}ms]"

        fj = len(Obs.Renderers._section_fields(jit.proc4))
        fp = len(Obs.Renderers._section_fields(pri.proc4))
        fs = len(Obs.Renderers._section_fields(sec.proc4))
        ex = max(1, time.perf_counter_ns() - tel.phase_entry_ns["Procedure4_LogAndRelease"]) / 1_000_000
        tj = (ex*fj)/max(1,sum(ord(c) for c in Core.obj_uid(jit))+fj)
        tp = (ex*fp)/max(1,sum(ord(c) for c in Core.obj_uid(pri))+fp)
        ts = (ex*fs)/max(1,sum(ord(c) for c in Core.obj_uid(sec))+fs)
        jit.proc4.magnitude_trajectory = tj
        pri.proc4.magnitude_trajectory = tp
        sec.proc4.magnitude_trajectory = ts
        tel.record_tri_metrics("Procedure4_LogAndRelease", fj, fp, fs, tj, tp, ts)

        HFEngine.fire("Proc4", "on_exit", jit, pri, sec, tel)
        Log.trace("4.E", "HF on_exit complete — tandem lock broadcast to both probes", jit, pri, sec, tel)

        Con._step("4.4", "Exit markers + del jit")
        ms   = Core.ms_now()
        exit_uid = UIDGen.next("EXIT-Proc4")
        pri.proc4.exit_marker    = f"[ {ms} ] PKG001 exit_uid={exit_uid}"
        sec.proc4.exit_marker    = f"[ {ms} ] PKG002 exit_uid={exit_uid}"
        jit.proc4.phase_exit_uid = exit_uid
        pri.proc4.phase_exit_uid = exit_uid
        sec.proc4.phase_exit_uid = exit_uid

        Log.trace("4.4", "Releasing JITCompileObject", jit, pri, sec, tel)
        del jit;  gc.collect()
        Con._field("result", "JITCompileObject released — PKG001 and PKG002 returned")

        tel.phase_exit("Procedure4_LogAndRelease")
        Con._procedure_footer(4, tel, "Procedure4_LogAndRelease", exit_uid)
        return pri, sec

JITPipelineArchitecture.Execution.Procedures.Proc4 = DataAcquisitionProcedure4

In [ ]:
# CELL 7: FACTORY ORCHESTRATOR ENGINE
class TriProbeFactoryEngine:
    @staticmethod
    def run_data_acquisition_factory(primary_probe, secondary_probe):
        Arch = JITPipelineArchitecture
        Con  = Arch.Observability.Console
        P    = Arch.Execution.Procedures

        tel = Arch.Data.Telemetry.PipelineTelemetry()
        tel.pipeline_start_ns  = time.perf_counter_ns()
        tel.pipeline_uid       = UIDGen.next("PIPELINE")
        primary_probe.telemetry = tel

        print(f"\n┌────────────────────────────────────────────────────────────────")
        print(f"│  FACTORY  ·  run_data_acquisition_factory")
        print(f"│  {Arch.Core.ms_now()}\n│")
        print(f"│  PRI received : {Arch.Core.obj_uid(primary_probe)}")
        print(f"│  SEC received : {Arch.Core.obj_uid(secondary_probe)}")
        jit = Arch.Data.Models.JITCompileObject()
        jit.payload = PayloadSection()
        print(f"│  JIT init     : {Arch.Core.obj_uid(jit)}")
        print(f"└────────────────────────────────────────────────────────────────\n")

        Arch.Observability.FiniteLogger.trace("0.0", "Factory entry", jit, primary_probe, secondary_probe, tel)

        jit, primary_probe, secondary_probe = P.Proc1.execute(jit, primary_probe, secondary_probe, tel)

        # ══ HF EXCHANGE · TRANSITION Proc1→Proc2 ══════════════════════════════
        HFEngine.fire("Proc1", "transition", jit, primary_probe, secondary_probe, tel)
        Con._inter_procedure(jit, primary_probe, secondary_probe, "Procedure 1", "Procedure 2")

        jit, primary_probe, secondary_probe = P.Proc2.execute(jit, primary_probe, secondary_probe, tel)

        # ══ HF EXCHANGE · TRANSITION Proc2→Proc3 ══════════════════════════════
        HFEngine.fire("Proc2", "transition", jit, primary_probe, secondary_probe, tel)
        Con._inter_procedure(jit, primary_probe, secondary_probe, "Procedure 2", "Procedure 3")

        jit, primary_probe, secondary_probe = P.Proc3.execute(jit, primary_probe, secondary_probe, tel)

        # ══ HF EXCHANGE · TRANSITION Proc3→Proc4 ══════════════════════════════
        HFEngine.fire("Proc3", "transition", jit, primary_probe, secondary_probe, tel)
        Con._inter_procedure(jit, primary_probe, secondary_probe, "Procedure 3", "Procedure 4")

        # ══ HF EXCHANGE · TRANSITION Proc4 pre-release ════════════════════════
        # NOTE: Proc4 transition fires here before JIT is passed in,
        # giving both probes a final corridor exchange before JIT deletion.
        HFEngine.fire("Proc4", "transition", jit, primary_probe, secondary_probe, tel)

        primary_probe, secondary_probe = P.Proc4.execute(jit, primary_probe, secondary_probe, tel)

        print(f"\n┌────────────────────────────────────────────────────────────────")
        print(f"│  FACTORY  ·  Pipeline complete")
        print(f"│  {Arch.Core.ms_now()}")
        print(f"│  Pipeline UID  : {tel.pipeline_uid}")
        print(f"│  JIT destroyed  |  Probes returned")
        print(f"│  PRI {Arch.Core.obj_uid(primary_probe)}")
        print(f"│  SEC {Arch.Core.obj_uid(secondary_probe)}")
        print(f"└────────────────────────────────────────────────────────────────\n")

        return primary_probe, secondary_probe

JITPipelineArchitecture.Execution.Factory = TriProbeFactoryEngine

In [ ]:
# CELL 8: INVOCATION, TELEMETRY & VISUALIZATION
if __name__ == "__main__":
    Arch = JITPipelineArchitecture

    print(f"\n┌────────────────────────────────────────────────────────────────")
    print(f"│  INVOKE  ·  Denver Hub Package Pipeline")
    print(f"│  {Arch.Core.ms_now()}\n│")

    # ── PKG001 → Primary Probe ────────────────────────────────────────────
    pri_probe     = Arch.Data.Models.PrimaryProbeObject()
    pri_probe.uid = UIDGen.next("PKG001")
    pri_probe.caller.request_label = "PKG001"
    pri_probe.caller.uid           = pri_probe.uid

    # HF numeric inputs (separate property space)
    pri_probe.payload.hf_value_a = 2
    # TODO: try pri_probe.payload.hf_value_a = 5

    # Package scan features (additional feature space)
    sf = pri_probe.payload.scan
    sf.package_id          = "PKG001"
    sf.inbound_scan_id     = PKG001_SCANS[0]["scan_id"];  sf.inbound_station    = PKG001_SCANS[0]["station"]
    sf.inbound_timestamp   = PKG001_SCANS[0]["timestamp"];sf.inbound_offset_min = PKG001_SCANS[0]["offset_min"]
    sf.initial_scan_id     = PKG001_SCANS[1]["scan_id"];  sf.initial_station    = PKG001_SCANS[1]["station"]
    sf.initial_timestamp   = PKG001_SCANS[1]["timestamp"];sf.initial_offset_min = PKG001_SCANS[1]["offset_min"]
    sf.belt_scan_id        = PKG001_SCANS[2]["scan_id"];  sf.belt_station       = PKG001_SCANS[2]["station"]
    sf.belt_timestamp      = PKG001_SCANS[2]["timestamp"];sf.belt_offset_min    = PKG001_SCANS[2]["offset_min"]
    sf.conv_scan_id        = PKG001_SCANS[3]["scan_id"];  sf.conv_station       = PKG001_SCANS[3]["station"]
    sf.conv_timestamp      = PKG001_SCANS[3]["timestamp"];sf.conv_offset_min    = PKG001_SCANS[3]["offset_min"]
    sf.outbound_scan_id    = PKG001_SCANS[4]["scan_id"];  sf.outbound_station   = PKG001_SCANS[4]["station"]
    sf.outbound_timestamp  = PKG001_SCANS[4]["timestamp"];sf.outbound_offset_min= PKG001_SCANS[4]["offset_min"]

    # ── PKG002 → Secondary Probe ──────────────────────────────────────────
    sec_probe     = Arch.Data.Models.SecondaryProbeObject()
    sec_probe.uid = UIDGen.next("PKG002")
    sec_probe.caller.request_label = "PKG002"
    sec_probe.caller.uid           = sec_probe.uid

    # HF numeric inputs
    sec_probe.payload.hf_value_b = 3   # Proc1: 2×3=6
    sec_probe.payload.hf_value_c = 4   # Proc2: 6×4=24
    sec_probe.payload.hf_value_d = 2   # Proc3: 24×2=48
    # TODO: try sec_probe.payload.hf_value_b = 10

    # Package scan features
    sf = sec_probe.payload.scan
    sf.package_id          = "PKG002"
    sf.inbound_scan_id     = PKG002_SCANS[0]["scan_id"];  sf.inbound_station    = PKG002_SCANS[0]["station"]
    sf.inbound_timestamp   = PKG002_SCANS[0]["timestamp"];sf.inbound_offset_min = PKG002_SCANS[0]["offset_min"]
    sf.initial_scan_id     = PKG002_SCANS[1]["scan_id"];  sf.initial_station    = PKG002_SCANS[1]["station"]
    sf.initial_timestamp   = PKG002_SCANS[1]["timestamp"];sf.initial_offset_min = PKG002_SCANS[1]["offset_min"]
    sf.belt_scan_id        = PKG002_SCANS[2]["scan_id"];  sf.belt_station       = PKG002_SCANS[2]["station"]
    sf.belt_timestamp      = PKG002_SCANS[2]["timestamp"];sf.belt_offset_min    = PKG002_SCANS[2]["offset_min"]
    sf.conv_scan_id        = PKG002_SCANS[3]["scan_id"];  sf.conv_station       = PKG002_SCANS[3]["station"]
    sf.conv_timestamp      = PKG002_SCANS[3]["timestamp"];sf.conv_offset_min    = PKG002_SCANS[3]["offset_min"]
    sf.outbound_scan_id    = PKG002_SCANS[4]["scan_id"];  sf.outbound_station   = PKG002_SCANS[4]["station"]
    sf.outbound_timestamp  = PKG002_SCANS[4]["timestamp"];sf.outbound_offset_min= PKG002_SCANS[4]["offset_min"]

    print(f"│  PKG001 uid={pri_probe.uid}")
    print(f"│    HF inputs   : a={pri_probe.payload.hf_value_a}")
    print(f"│    Scan events : {len(PKG001_SCANS)} loaded")
    print(f"│  PKG002 uid={sec_probe.uid}")
    print(f"│    HF inputs   : b={sec_probe.payload.hf_value_b}  c={sec_probe.payload.hf_value_c}  d={sec_probe.payload.hf_value_d}")
    print(f"│    Scan events : {len(PKG002_SCANS)} loaded")
    print(f"└────────────────────────────────────────────────────────────────\n")

    res_pri, res_sec = Arch.Execution.Factory.run_data_acquisition_factory(pri_probe, sec_probe)

    tel = res_pri.telemetry

    print(f"\n┌────────────────────────────────────────────────────────────────")
    print(f"│  RESULT VALIDATION")
    print(f"│  PKG001 exit   : {res_pri.proc4.exit_marker}")
    print(f"│  PKG002 exit   : {res_sec.proc4.exit_marker}")
    print(f"│  HF chain      : 2×3 = {res_pri.payload.hf_result_p1}")
    print(f"│             → ×4 = {res_pri.payload.hf_result_p2}")
    print(f"│             → ×2 = {res_pri.payload.hf_result_p3}")
    print(f"└────────────────────────────────────────────────────────────────\n")

    if tel:
        logger = logging.getLogger("ColabFiniteLogger")
        logger.setLevel(logging.INFO)
        if not logger.handlers:
            logger.addHandler(logging.StreamHandler(sys.stdout))

        # 1. Finite telemetry summary
        Arch.Observability.Renderers.render_telemetry_summary(logger, tel)

        # 2. Chronological card log
        JITPipelineObservability.CardLog.render_all(tel, res_pri, res_sec, logger)

        # 3. Animated 3D visualization
        Arch.Observability.Visualizations.render_animated_telemetry(tel, res_pri, res_sec)